# Eksplorasi Data Mentah (Raw Data Exploration & Justification)

Notebook ini menganalisis kondisi riil dari dataset mentah (*raw data*) sebelum dilakukan pemrosesan. Analisis ini mendasari keputusan desain pipeline pemrosesan data, ekstraksi fitur, dan pembersihan teks.

## 1. Meninjau File Dataset Mentah

Dataset mentah tersimpan dalam folder `data/raw/`. Mari kita daftarkan seluruh file CSV yang ada.

In [8]:
import os
import pandas as pd
import numpy as np

raw_dir = r'../data/raw'
csv_files = [f for f in os.listdir(raw_dir) if f.startswith('dataset-') and f.endswith('.csv')]
print('Daftar File CSV Mentah:')
for f in csv_files:
    size_kb = os.path.getsize(os.path.join(raw_dir, f)) / 1024
    print(f'- {f} ({size_kb:.2f} KB)')

Daftar File CSV Mentah:
- dataset-ayam.csv (1647.74 KB)
- dataset-ikan.csv (1526.38 KB)
- dataset-kambing.csv (1810.68 KB)
- dataset-sapi.csv (1758.29 KB)
- dataset-tahu.csv (1485.22 KB)
- dataset-telur.csv (1370.14 KB)
- dataset-tempe.csv (1418.07 KB)
- dataset-udang.csv (1471.80 KB)


**Analisis Awal:**
- Terdapat 8 file CSV berbeda yang dikelompokkan berdasarkan bahan protein utamanya (ayam, ikan, kambing, sapi, tahu, telur, tempe, udang).
- **Justifikasi Merging:** Pengguna memerlukan sistem pencarian resep universal. Kita harus menyatukan ke-8 file ini ke dalam satu DataFrame utama dan menambahkan fitur `category` berdasarkan nama file asal agar pengguna tetap dapat memfilter berdasarkan jenis protein.

## 2. Pemeriksaan Struktur dan Kolom Raw Dataset

Mari kita muat salah satu dataset (misalnya: `dataset-ayam.csv`) untuk melihat bentuk data aslinya.

In [9]:
sample_df = pd.read_csv(os.path.join(raw_dir, 'dataset-ayam.csv'))
print(f'Dimensi dataset ayam: {sample_df.shape}')
sample_df.info()

Dimensi dataset ayam: (1916, 5)
<class 'pandas.DataFrame'>
RangeIndex: 1916 entries, 0 to 1915
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   Title        1916 non-null   str  
 1   Ingredients  1901 non-null   str  
 2   Steps        1901 non-null   str  
 3   Loves        1916 non-null   int64
 4   URL          1916 non-null   str  
dtypes: int64(1), str(4)
memory usage: 1.7 MB


In [10]:
print('5 Baris Pertama:')
sample_df.head(5)

5 Baris Pertama:


,Title,Ingredients,Steps,Loves,URL
0,Ayam Woku Manado,1 Ekor Ayam Kampung (potong 12)--2 Buah Jeruk ...,Cuci bersih ayam dan tiriskan. Lalu peras jeru...,1,/id/resep/4473027-ayam-woku-manado
1,Ayam goreng tulang lunak,1 kg ayam (dipotong sesuai selera jangan kecil...,"Haluskan bumbu2nya (BaPut, ketumbar, kemiri, k...",1,/id/resep/4471956-ayam-goreng-tulang-lunak
2,Ayam cabai kawin,1/4 kg ayam--3 buah cabai hijau besar--7 buah ...,Panaskan minyak di dalam wajan. Setelah minyak...,2,/id/resep/4473057-ayam-cabai-kawin
3,Ayam Geprek,250 gr daging ayam (saya pakai fillet)--Secuku...,Goreng ayam seperti ayam krispi--Ulek semua ba...,10,/id/resep/4473023-ayam-geprek
4,Minyak Ayam,400 gr kulit ayam & lemaknya--8 siung bawang p...,Cuci bersih kulit ayam. Sisihkan--Ambil 50 ml ...,4,/id/resep/4427438-minyak-ayam


**Keterbatasan Metadata Asli:**
1. Kolom yang tersedia hanya: `Title`, `Ingredients`, `Steps`, `Loves`, dan `URL`.
2. **Tidak ada kolom Waktu Memasak (`cooking_time`):** Padahal waktu memasak adalah filter esensial bagi pengguna yang sedang terburu-buru.
3. **Tidak ada kolom Tingkat Kepedasan (`spice_level`):** Sangat penting bagi kuliner Indonesia untuk membedakan makanan pedas dan ramah anak.
4. **Tidak ada kolom Tipe Diet (`diet_type`):** Pengguna dengan preferensi vegetarian tidak memiliki cara menyaring menu hewani vs nabati.
5. **Kolom URL:** Berisi tautan web yang tidak diperlukan dalam pemrosesan rekomendasi (sebaiknya dihapus/di-drop).

## 3. Deteksi Data Kosong (Null) dan Tidak Valid (Invalid Data Detection)

Sebelum melakukan merging, kita perlu menganalisis baris yang rusak atau kosong dalam seluruh dataset raw. Baris korup didefinisikan sebagai resep yang memiliki judul `[Notitle]`, atau resep yang kolom `Ingredients` atau `Steps`-nya bernilai kosong (NaN).

In [11]:
dfs = []
for f in csv_files:
    df_temp = pd.read_csv(os.path.join(raw_dir, f))
    df_temp['source_file'] = f
    dfs.append(df_temp)
raw_merged = pd.concat(dfs, ignore_index=True)

# Deteksi baris yang tidak valid
is_notitle = raw_merged['Title'].astype(str).str.lower().str.strip() == '[notitle]'
is_null_title = raw_merged['Title'].isnull()
is_null_ingredients = raw_merged['Ingredients'].isnull()
is_null_steps = raw_merged['Steps'].isnull()

invalid_rows = raw_merged[is_notitle | is_null_title | is_null_ingredients | is_null_steps]
total_invalid = len(invalid_rows)

print(f'Total baris tidak valid terdeteksi: {total_invalid}')
print('\nContoh baris tidak valid yang ditemukan:')
invalid_rows[['Title', 'Ingredients', 'Steps', 'source_file']].head(10)

Total baris tidak valid terdeteksi: 48

Contoh baris tidak valid yang ditemukan:


,Title,Ingredients,Steps,source_file
255,[Notitle],NaN,NaN,dataset-ayam.csv
258,[Notitle],NaN,NaN,dataset-ayam.csv
298,[Notitle],NaN,NaN,dataset-ayam.csv
302,[Notitle],NaN,NaN,dataset-ayam.csv
464,[Notitle],NaN,NaN,dataset-ayam.csv
521,[Notitle],NaN,NaN,dataset-ayam.csv
523,[Notitle],NaN,NaN,dataset-ayam.csv
524,[Notitle],NaN,NaN,dataset-ayam.csv
835,[Notitle],NaN,NaN,dataset-ayam.csv
849,[Notitle],NaN,NaN,dataset-ayam.csv


**Justifikasi Pembersihan Data Corrupt:**
- Kita menemukan beberapa baris data korup (seperti `[Notitle]` dan nilai NaN). Baris-baris ini tidak memberikan informasi berguna bagi model TF-IDF maupun bagi pengguna Streamlit.
- **Rencana Tindakan:** Kita harus secara ketat membuang (*drop*) baris-baris yang tidak valid ini di dalam pipeline preprocessing agar dataset akhir benar-benar bersih dan bebas dari error.

## 4. Investigasi Konten Teks (Ingredients & Steps)

Mari kita teliti struktur teks pada kolom `Ingredients` dan `Steps` untuk memahami kebisingan data (*noise*).

In [12]:
# Tampilkan contoh baris pertama dari ayam
row = sample_df.iloc[0]
print('=== CONTOH INGREDIENTS RAW ===')
print(row['Ingredients'])
print('\n=== CONTOH STEPS RAW ===')
print(row['Steps'])

=== CONTOH INGREDIENTS RAW ===
1 Ekor Ayam Kampung (potong 12)--2 Buah Jeruk Nipis--2 Sdm Garam--3 Ruas Kunyit--7 Bawang Merah--7 Bawang Putih--10 Cabe Merah--10 Cabe Rawit Merah (sesuai selera)--3 Butir Kemiri--2 Batang Sereh--2 Lembar Daun Salam--2 Ikat Daun Kemangi--Penyedap Rasa--1 1/2 Gelas Air--

=== CONTOH STEPS RAW ===
Cuci bersih ayam dan tiriskan. Lalu peras jeruk nipis (kalo gak ada jeruk nipis bisa pake cuka) dan beri garam. Aduk hingga merata dan diamkan selama 5 menit, biar ayam gak bau amis.--Goreng ayam tersebut setengah matang, lalu tiriskan--Haluskan bumbu menggunakan blender. Bawang merah, bawang putih, cabe merah, cabe rawit, kemiri dan kunyit. Oh iya kasih minyak sedikit yaa biar bisa di blender. Untuk sereh nya di geprek aja terus di buat simpul.--Setelah bumbu di haluskan barulah di tumis. Jangan lupa sereh dan daun salamnya juga ikut di tumis. Di tumis sampai berubah warna ya 👌--Masukan ayam yang sudah di goreng setengah matang ke dalam bumbu yang sudah di tumis

**Temuan Kunci dari Investigasi Teks:**
1. **Pemisah Unsur (`--`):** Baik bahan-bahan maupun langkah-langkah dipisahkan oleh tanda dua strip (`--`).
2. **Stopwords & Takaran:** Kolom bahan mengandung banyak kata takaran seperti *'secukupnya'*, *'sendok makan'*, *'sdm'*, *'siung'*, *'bks'*, *'gram'* serta kata hubung *'dan'*, *'yang'*, *'di'*. 
   - *Justifikasi Stopwords:* Kata-kata takaran ini muncul di hampir setiap resep. Jika tidak dihapus, TF-IDF akan menganggap kata-kata ini penting, yang akan merusak hasil pencarian kecocokan bahan. Kita harus menghapusnya dalam proses modeling.
3. **Informasi Waktu Tersembunyi:** Informasi waktu memasak sering ditulis secara eksplisit di dalam teks langkah-langkah (misal: *'diamkan selama 5 menit'*, *'masak lagi sekitar 10 menit'*).
   - *Justifikasi Regex Waktu:* Kita dapat menggunakan Regular Expression untuk memindai angka yang diikuti kata 'menit', 'mnt', atau 'jam' dari teks langkah-langkah masakan untuk membentuk fitur `cooking_time` secara otomatis.
4. **Petunjuk Hewani/Pedas:** Kita bisa memindai kata kunci seperti 'ayam', 'daging', 'sapi', 'ikan' untuk mengklasifikasi tipe diet. Begitu pula kata kunci 'cabai', 'cabe', 'lada', 'rawit' dapat digunakan untuk mengklasifikasi tingkat kepedasan secara heuristik.

## 5. Kesimpulan Rencana Pemrosesan Data

Berdasarkan analisis kondisi riil di atas, pendekatan kita dirancang sebagai berikut:
1. **Merge & Kategori:** Satukan 8 file CSV dan buat kolom `category` dari nama file.
2. **Pembersihan Data Corrupt:** Memfilter keluar data dengan `[Notitle]`, NaN, atau string kosong pada kolom utama.
3. **Ekstraksi Fitur:** 
   - Ekstrak `cooking_time` menggunakan regex pencarian durasi di kolom `Steps`.
   - Bentuk `spice_level` dan `diet_type` dengan pemindaian kata kunci di kolom `Ingredients`.
4. **Pembersihan Teks Pendekatan Dual-Kolom:**
   - Lakukan pembersihan intensif (lowercase, hapus tanda baca, hapus stopwords masakan) pada kolom `Title`, `Ingredients`, dan `Steps` untuk pencarian TF-IDF.
   - Simpan teks asli ke kolom pendamping `*_Display` agar UI Streamlit tetap menampilkan takaran dan instruksi langkah-langkah yang rapi untuk pengguna.